In [1]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv('data/processed/amr_cohort.csv')

print(f"Raw cohort shape: {df.shape}")
print(f"\nColumn names:")
print(df.columns.tolist())
print(f"\nData types:")
print(df.dtypes)

Raw cohort shape: (8476, 12)

Column names:
['nct_id', 'brief_title', 'overall_status', 'phase', 'start_date', 'completion_date', 'enrollment', 'sponsor_name', 'lead_or_collaborator', 'start_year', 'who_gap_period', 'condition']

Data types:
nct_id                      str
brief_title                 str
overall_status              str
phase                       str
start_date                  str
completion_date             str
enrollment              float64
sponsor_name                str
lead_or_collaborator        str
start_year              float64
who_gap_period              str
condition                   str
dtype: object


Need to convert dates to datetime and trial duration to months by dividing days by average month length 30.44. This serves as a normalization factor for determining how long trials ran. 

In [9]:
# Convert dates to datetime
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['completion_date'] = pd.to_datetime(df['completion_date'], errors='coerce')

# Calculate trial duration in months (via days / 30.44)
df['duration_months'] = (
    (df['completion_date'] - df['start_date']).dt.days / 30.44
).round(1)

print(f"Date range: {df['start_date'].min()} to {df['start_date'].max()}")
print(f"\nMissing start dates: {df['start_date'].isna().sum()}")
print(f"Missing completion dates: {df['completion_date'].isna().sum()}")
print(f"\nDuration stats (months):")
print(df['duration_months'].describe())

Date range: 2004-01-31 00:00:00 to 2026-06-01 00:00:00

Missing start dates: 20
Missing completion dates: 60

Duration stats (months):
count    8416.000000
mean       46.755466
std        29.846390
min         0.000000
25%        24.000000
50%        39.900000
75%        66.100000
max       177.000000
Name: duration_months, dtype: float64


Clean Phase Problem:

In [2]:
# Replace nulls stored as strings
df['phase'] = df['phase'].replace({'phase': np.nan, 'NA': np.nan})

# Standardize phase names
phase_mapping = {
    'PHASE1': 'Phase 1',
    'EARLY_PHASE1': 'Phase 1',
    'PHASE1/PHASE2': 'Phase 1/2',
    'PHASE2': 'Phase 2',
    'PHASE2/PHASE3': 'Phase 2/3',
    'PHASE3': 'Phase 3',
    'PHASE4': 'Phase 4'
}

df['phase'] = df['phase'].map(phase_mapping)

print("Phase distribution after cleaning:")
print(df['phase'].value_counts(dropna=False))

Phase distribution after cleaning:
phase
NaN          5260
Phase 3       796
Phase 2/3     620
Phase 1       584
Phase 2       560
Phase 4       548
Phase 1/2     108
Name: count, dtype: int64


In [3]:
print("Raw status values:")
print(df['overall_status'].value_counts(dropna=False))


# Standardize status into broader categories
status_mapping = {
    'COMPLETED': 'Completed',
    'TERMINATED': 'Terminated',
    'WITHDRAWN': 'Withdrawn',
    'SUSPENDED': 'Suspended',
    'RECRUITING': 'Active',
    'ACTIVE_NOT_RECRUITING': 'Active',
    'ENROLLING_BY_INVITATION': 'Active',
    'NOT_YET_RECRUITING': 'Active',
    'UNKNOWN': np.nan
}

df['status_clean'] = df['overall_status'].map(status_mapping)

print("\nCleaned status distribution:")
print(df['status_clean'].value_counts(dropna=False))

Raw status values:
overall_status
COMPLETED                  4448
RECRUITING                 1344
UNKNOWN                    1164
ACTIVE_NOT_RECRUITING       508
TERMINATED                  436
NOT_YET_RECRUITING          404
WITHDRAWN                   120
ENROLLING_BY_INVITATION      44
AVAILABLE                     8
Name: count, dtype: int64

Cleaned status distribution:
status_clean
Completed     4448
Active        2300
NaN           1172
Terminated     436
Withdrawn      120
Name: count, dtype: int64


Some government trials can have extremely large enrollment numbers. We are ensuring that these do not show up for visualizations to prevent skewed data.

In [5]:
# Enrollment should be numeric
df['enrollment'] = pd.to_numeric(df['enrollment'], errors='coerce')

# Flag extreme outliers for transparency
print(f"Enrollment stats:")
print(df['enrollment'].describe())
print(f"\nTrials with enrollment > 10000: {(df['enrollment'] > 10000).sum()}")
print(f"Trials with enrollment = 0: {(df['enrollment'] == 0).sum()}")

# Cap extreme outliers at 99th percentile for visualization purposes
# Keep original column intact
enrollment_cap = df['enrollment'].quantile(0.99)
df['enrollment_capped'] = df['enrollment'].clip(upper=enrollment_cap)

print(f"\n99th percentile cap: {enrollment_cap:.0f}")

Enrollment stats:
count    8.460000e+03
mean     2.029960e+04
std      2.478158e+05
min      0.000000e+00
25%      1.000000e+02
50%      3.060000e+02
75%      1.000000e+03
max      3.300000e+06
Name: enrollment, dtype: float64

Trials with enrollment > 10000: 296
Trials with enrollment = 0: 120

99th percentile cap: 25732


Clean Sponsor Names and group them by Academic/Research vs Government vs Industry.


In [7]:
print("Sponsor lead/collaborator split:")
print(df['lead_or_collaborator'].value_counts(dropna=False))

# Classify sponsor type broadly
def classify_sponsor(name):
    if pd.isna(name):
        return np.nan
    name_lower = name.lower()
    if any(term in name_lower for term in ['university', 'college', 'institute', 
                                            'hospital', 'school', 'academic',
                                            'foundation', 'centre', 'center']):
        return 'Academic/Research'
    elif any(term in name_lower for term in ['nih', 'cdc', 'fda', 'who', 
                                              'national', 'federal', 'ministry',
                                              'government', 'dept', 'department']):
        return 'Government'
    else:
        return 'Industry'

df['sponsor_category'] = df['sponsor_name'].apply(classify_sponsor)

print("\nSponsor category distribution:")
print(df['sponsor_category'].value_counts(dropna=False))

Sponsor lead/collaborator split:
lead_or_collaborator
collaborator    4884
lead            3592
Name: count, dtype: int64

Sponsor category distribution:
sponsor_category
Academic/Research    5420
Industry             2812
Government            244
Name: count, dtype: int64


In [10]:
# Fix who_gap_period for rows with missing start dates
df['who_gap_period'] = np.where(
    df['start_date'].isna(),
    np.nan,
    df['who_gap_period']
)

# Ensure start_year is integer
df['start_year'] = df['start_date'].dt.year

print("WHO GAP period split:")
print(df['who_gap_period'].value_counts(dropna=False))

print(f"\nYear range: {df['start_year'].min()} to {df['start_year'].max()}")

WHO GAP period split:
who_gap_period
Post-WHO GAP    6620
Pre-WHO GAP     1836
NaN               20
Name: count, dtype: int64

Year range: 2004.0 to 2026.0


In [11]:
print(f"Shape before deduplication: {df.shape}")

# A trial can appear multiple times due to multiple conditions matching
# Keep one row per trial for trial-level analysis
df_trials = df.drop_duplicates(subset='nct_id')

print(f"Shape after deduplication (trial level): {df_trials.shape}")
print(f"Duplicates removed: {len(df) - len(df_trials)}")


print(f"Total rows: {len(df)}")
print(f"Unique trials: {df['nct_id'].nunique()}")
print(f"Average conditions per trial: {len(df) / df['nct_id'].nunique():.1f}")

Shape before deduplication: (8476, 16)
Shape after deduplication (trial level): (313, 16)
Duplicates removed: 8163
Total rows: 8476
Unique trials: 313
Average conditions per trial: 27.1


Final save and creation of condition-csv and trial-csv for different analyses


In [ ]:
print("=== FINAL CLEANED COHORT SUMMARY ===")
print(f"\nTotal unique trials: {len(df_trials)}")
print(f"Date range: {df_trials['start_date'].min().year} to {df_trials['start_date'].max().year}")
print(f"\nWHO GAP split:")
print(df_trials['who_gap_period'].value_counts(dropna=False))
print(f"\nPhase distribution:")
print(df_trials['phase'].value_counts(dropna=False))
print(f"\nSponsor category:")
print(df_trials['sponsor_category'].value_counts(dropna=False))
print(f"\nStatus distribution:")
print(df_trials['status_clean'].value_counts(dropna=False))

# Save both versions
# df = full with duplicate conditions for conditional-level analysis
# df_trials = deduplicated trial-level dataset

df.to_csv('data/processed/amr_cohort_clean.csv', index=False)
df_trials.to_csv('data/processed/amr_trials_clean.csv', index=False)

print("\nSaved:")
print("  data/processed/amr_cohort_clean.csv  (condition level)")
print("  data/processed/amr_trials_clean.csv  (trial level, deduplicated)")

=== FINAL CLEANED COHORT SUMMARY ===

Total unique trials: 313
Date range: 2004 to 2026

WHO GAP split:
who_gap_period
Post-WHO GAP    239
Pre-WHO GAP      71
NaN               3
Name: count, dtype: int64

Phase distribution:
phase
NaN          193
Phase 4       31
Phase 2       29
Phase 3       25
Phase 1       18
Phase 2/3     12
Phase 1/2      5
Name: count, dtype: int64

Sponsor category:
sponsor_category
Academic/Research    194
Industry             115
Government             4
Name: count, dtype: int64

Status distribution:
status_clean
Completed     149
Active         76
NaN            59
Terminated     20
Withdrawn       9
Name: count, dtype: int64

Saved:
  data/processed/amr_cohort_clean.csv  (condition level)
  data/processed/amr_trials_clean.csv  (trial level, deduplicated)
